# Laboratorium 11 — projektowanie i ocena niewielkiego pipeline'u inferencji LLM

Materiał towarzyszący [`README.md`](README.md). Pipeline: prompt → tokenizacja →
model → generowanie → dekodowanie → pomiar → ocena.


In [ ]:
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
torch.set_num_threads(4)
device = torch.device("cpu")
print(f"Uzywane urzadzenie: {device}")

MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
REVISION = "12fd25f77366fa6b3b4b768ec3050bf629380bac"


## 1. Model i tokenizer (gotowe + krótkie zadanie)


In [ ]:
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=REVISION, dtype=torch.float32)
model.eval()
print(f"load time: {time.time() - t0:.2f}s")
print(f"liczba parametrow: {sum(p.numel() for p in model.parameters()):,}")


In [ ]:
# TODO: wypisz model.config.num_attention_heads i model.config.num_key_value_heads
# Czy sa rowne? Co to oznacza dla kosztu pamieci KV cache? (zobacz wyklad)
raise NotImplementedError("TODO: sprawdzenie glow Q vs KV")


## 2. Przygotowanie promptów — Twoje zadanie

Lista słowników `{"id", "typ", "prompt", "expected"}` — `expected=None` dla
zadań jakościowych (parafraza, kontynuacja). Zapisz `expected` **przed**
wygenerowaniem odpowiedzi.


In [ ]:
# TODO: zbuduj liste 3-5 slownikow {"id","typ","prompt","expected"}, pokrywajacych
# co najmniej: ekstrakcje informacji, przeksztalcenie formatu, i jedno zadanie
# JAKOSCIOWE (parafraza albo kontynuacja, expected=None)
prompts = []
raise NotImplementedError("TODO: przygotowanie promptow")


## 3. Chat template — Twoje zadanie

**Uwaga API:** `apply_chat_template(..., tokenize=True, return_tensors="pt")`
**bez** `return_dict=True` może zwrócić inny typ niż oczekujesz w Twojej
wersji `transformers` — użyj `return_dict=True` jawnie.


In [ ]:
def format_prompt(prompt_text):
    # TODO:
    # 1. zbuduj messages = [{"role": "user", "content": prompt_text}]
    # 2. zwroc tokenizer.apply_chat_template(messages, tokenize=True,
    #    add_generation_prompt=True, return_tensors="pt", return_dict=True)
    raise NotImplementedError("TODO: format_prompt")


**Weryfikacja (gotowa asercja): zdekodowany początek sformatowanego promptu
musi zaczynać się od tokenu specjalnego (BOS/`<|im_start|>`), nie od treści.**


In [ ]:
check_inputs = format_prompt("test")  # noqa: F821 - zdefiniowane w sekcji 3
start_decoded = tokenizer.decode(check_inputs["input_ids"][0][:1])
print(f"pierwszy token sformatowanego promptu: {start_decoded!r}")
assert start_decoded == tokenizer.bos_token, (
    f"BLAD: oczekiwano {tokenizer.bos_token!r} jako pierwszy token, dostano {start_decoded!r}"
)
print("OK: format_prompt dodaje token specjalny na starcie.")


## 4. Generowanie i poprawne dekodowanie — Twoje zadanie


In [ ]:
def generate_response(prompt_text, **gen_kwargs):
    # TODO:
    # 1. inputs = format_prompt(prompt_text)
    # 2. prompt_len = inputs["input_ids"].shape[1]
    # 3. out = model.generate(**inputs, pad_token_id=tokenizer.pad_token_id, **gen_kwargs)
    # 4. WYCIAGNIJ TYLKO nowo wygenerowane tokeny: out[0, prompt_len:] (NIE caly out!)
    # 5. zdekoduj (skip_special_tokens=True)
    # 6. zwroc (text, liczba_nowych_tokenow)
    raise NotImplementedError("TODO: generate_response")


**Weryfikacja (gotowa asercja): dla `do_sample=False`, dwa wywołania z tym
samym promptem muszą dać identyczny wynik.**


In [ ]:
r1, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
r2, _ = generate_response("What is the capital of France?", max_new_tokens=10, do_sample=False)  # noqa: F821
print(f"przebieg 1: {r1!r}")
print(f"przebieg 2: {r2!r}")
assert r1 == r2, "BLAD: greedy decoding powinien byc deterministyczny"
print("OK: deterministyczne powtorzenia sa identyczne.")


## 5. Deterministyczne generowanie i sampling — Twoje zadanie


In [ ]:
configs = [
    ("greedy", dict(do_sample=False)),
    ("sampling tau=0.7", dict(do_sample=True, temperature=0.7)),
    ("sampling tau=1.3", dict(do_sample=True, temperature=1.3)),
]

# TODO:
# dla kazdego promptu (sekcja 2) x kazdej konfiguracji (powyzej):
# 1. ustaw torch.manual_seed(0) PRZED kazdym wywolaniem (powtarzalnosc)
# 2. zmierz czas (time.time() przed/po) wywolania
#    generate_response(prompt, max_new_tokens=40, **kwargs)
# 3. zbierz wyniki (lista slownikow: id, strategy, text, n_new, time, tok_s)
results = []
raise NotImplementedError("TODO: generowanie dla wszystkich promptow x strategii")


## 6. Pomiar czasu i przepustowości (gotowe — już zebrane w sekcji 5)

**Zadanie:** wypisz tabelę tokenów/sekundę dla każdej kombinacji (prompt,
strategia) z `results`.


In [ ]:
# TODO: wypisz id, strategie i tok/s dla kazdego wpisu w `results`
raise NotImplementedError("TODO: tabela przepustowosci")


## 7. Wpływ długości promptu — Twoje zadanie

**Ważne:** użyj `min_new_tokens == max_new_tokens`, żeby wymusić **tę samą**
liczbę nowych tokenów dla obu długości promptu — inaczej różnice w czasie
generowania (np. wcześniejsze zatrzymanie przez EOS) zanieczyszczą pomiar
samego wpływu długości promptu (ta pułapka jest realna — sprawdzona przy
przygotowaniu tego laboratorium).


In [ ]:
short_prompt = "What is 2+2?"
long_prompt = ("Here is a lot of context to pad out the prompt. " * 30) + " What is 2+2?"

# TODO: dla obu promptow, zmierz czas generate_response z max_new_tokens=20 I
# min_new_tokens=20 (ta sama liczba nowych tokenow dla obu!), wypisz prompt_len i czas
raise NotImplementedError("TODO: wplyw dlugosci promptu")


## 8. KV cache — Twoje zadanie


In [ ]:
# TODO: dla use_cache=True i use_cache=False, zmierz czas generate_response
# ("Tell me about the solar system.", max_new_tokens=40, do_sample=False, use_cache=...)
raise NotImplementedError("TODO: KV cache")


## 9. Ocena jakości — Twoje zadanie

Dla każdego promptu z `expected != None`, sprawdź automatycznie, czy `expected`
(małymi literami) jest zawarte w odpowiedzi `greedy` (małymi literami). Dla
promptów jakościowych, napisz krótkie uzasadnienie w markdown.


In [ ]:
# TODO: dla kazdego promptu z expected != None, sprawdz czy
# p["expected"].lower() in <odpowiedz greedy tego promptu>.lower()
# wypisz id, expected, wynik sprawdzenia, i sama odpowiedz
raise NotImplementedError("TODO: automatyczna ocena")


**Zadanie (markdown):** dla promptu `paraphrase` i `continuation` (zadania
jakościowe), oceń **każdą** z trzech strategii (greedy, tau=0.7, tau=1.3) pod
względem: czy zachowuje znaczenie oryginału, czy jest kompletna, czy brzmi
naturalnie. Konkretne uzasadnienie, nie "wygląda dobrze".


_(Twoja odpowiedź tutaj.)_

## 10. Zadanie diagnostyczne

Poniższy kod **wygląda** poprawnie — nie rzuca wyjątku, generuje tekst — ale
ma błąd: pomylenie `max_length` z `max_new_tokens`.


In [ ]:
DESIRED_NEW_TOKENS = 40
diag_prompt = "Write a short poem about the ocean."
diag_inputs = format_prompt(diag_prompt)  # noqa: F821
diag_prompt_len = diag_inputs["input_ids"].shape[1]
print(f"prompt_len={diag_prompt_len}, chcemy {DESIRED_NEW_TOKENS} NOWYCH tokenow")

with torch.no_grad():
    out_buggy = model.generate(
        **diag_inputs, max_length=DESIRED_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
actual_new_buggy = out_buggy.shape[1] - diag_prompt_len
print(f"BuggyGeneration: faktycznie nowych tokenow = {actual_new_buggy} "
      f"(chcielismy {DESIRED_NEW_TOKENS}!)")
text_buggy = tokenizer.decode(out_buggy[0, diag_prompt_len:], skip_special_tokens=True)
print(f"tekst (uciety): {text_buggy!r}")


**Zadanie:**
1. Zdiagnozuj: co dokładnie policzył `max_length=40` tutaj, a co chcieliśmy?
2. Spróbuj tego samego kodu z **dłuższym** promptem (na tyle długim, by
   `prompt_len >= max_length`) — co się teraz dzieje? (Inne, głośniejsze
   zachowanie.)
3. Popraw kod.
4. Zweryfikuj poprawkę dla **obu** długości promptu.


In [ ]:
# TODO:
# 1. jako komentarz: zdiagnozuj dokladnie, co max_length liczy tutaj, a co chcielismy
# 2. sprobuj TEJ SAMEJ buggy konfiguracji z DLUZSZYM promptem (prompt_len >= max_length)
#    - co sie teraz dzieje? (podpowiedz: inny, glosniejszy blad)
# 3. napraw: uzyj max_new_tokens= zamiast max_length=
# 4. zweryfikuj POPRAWKE dla OBU dlugosci promptu (assert na liczbie nowych tokenow)
raise NotImplementedError("TODO: diagnoza i poprawka max_length/max_new_tokens")


## 11. Wnioski — Twoje zadanie


**1. Która strategia dekodowania dała najbardziej użyteczne wyniki dla zadań z jednoznaczną odpowiedzią? Czy sampling pomógł, zaszkodził, czy nie miał znaczenia?**

_(Twoja odpowiedź tutaj.)_

**2. Dla zadań jakościowych — czy wyższa temperatura dała subiektywnie lepsze wyniki? Czy różnorodność zależała od zadania?**

_(Twoja odpowiedź tutaj.)_

**3. Jaki jest zmierzony koszt pipeline'u? Jak zmienia się z długością promptu i z `use_cache`?**

_(Twoja odpowiedź tutaj.)_

**4. Jakie konkretne ograniczenia modelu/pipeline'u zaobserwowałaś/eś?**

_(Twoja odpowiedź tutaj.)_

**5. Jaki jest wniosek o kompromisie jakość/koszt/konfiguracja? Czy 'lepsza' konfiguracja zawsze oznacza lepszy wynik dla każdego typu zadania?**

_(Twoja odpowiedź tutaj.)_